# Tutorial 1: Basic EEG Pipeline — Age Prediction with LightGBM

This tutorial walks through a complete EEG analysis pipeline using the **EEGDash** package: from loading open-access EEG datasets, through preprocessing and feature extraction, to training a machine learning model that predicts participant age from resting-state EEG.

**What you will learn:**
- How to load EEG recordings from EEGDash with metadata filtering
- How to preprocess EEG using braindecode's `Preprocessor` API
- How to extract spectral and connectivity features with `FeatureExtractor`
- Why subject-level train/test splitting is essential to avoid data leakage
- How to fit and interpret a LightGBM regression model

**Required packages:** `eegdash`, `braindecode`, `lightgbm`, `shap`, `scikit-learn`, `pandas`, `numpy`, `matplotlib`

---
## 0. Exploring the Dataset

Before loading any data, it's worth getting a feel for what's actually in the two HBN releases. The `EEGDash` client gives you a lightweight way to query the metadata database directly — no files are downloaded, just JSON records from the server.

We'll check how many resting-state recordings are available, peek at a sample record, and look at the distribution of ages across subjects.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from eegdash import EEGDash

eeg = EEGDash()

# --- How many resting-state recordings exist across both releases? ---
n_total = eeg.count(
    {"dataset": {"$in": ["ds005505", "ds005506"]}, "task": "RestingState"}
)
print(f"Resting-state recordings (ds005505 + ds005506): {n_total}")

# --- Peek at a single record to see what fields are available ---
sample = eeg.find_one(dataset="ds005505", task="RestingState")
print("\nTop-level fields:", [k for k in sample.keys() if not k.startswith("_")])
print("\nSubject metadata (participant_tsv):")
for k, v in sample["participant_tsv"].items():
    print(f"  {k}: {v}")

# --- Fetch all records and inspect the age distribution ---
# Age and sex are stored in the nested participant_tsv field, not at the top level
records = eeg.find(
    {"dataset": {"$in": ["ds005505", "ds005506"]}, "task": "RestingState"}
)
meta_df = pd.DataFrame(records)
meta_df["age"] = meta_df["participant_tsv"].apply(lambda x: x.get("age"))
meta_df["sex"]  = meta_df["participant_tsv"].apply(lambda x: x.get("sex"))

print(f"\nTotal recordings : {len(meta_df)}")
print(f"Age range        : {meta_df['age'].min():.1f} – {meta_df['age'].max():.1f} years")
print(f"Sex breakdown    : {meta_df['sex'].value_counts().to_dict()}")

# Age distribution
fig, ax = plt.subplots(figsize=(7, 3))
meta_df["age"].astype(float).hist(bins=25, ax=ax, color="steelblue", edgecolor="white")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Number of recordings")
ax.set_title("Age distribution — HBN resting-state (R1 + R2)")
plt.tight_layout()
plt.show()

---
## 1. Data Loading

EEGDash provides a unified interface to open-access EEG datasets stored in BIDS format. We load both HBN releases (`ds005505` = R1, `ds005506` = R2) and filter to the `RestingState` task. Recordings are downloaded to a local cache on the first run — subsequent runs read directly from disk.

The `description_fields` argument controls which metadata columns are attached to each recording. Including `"age"` and `"sex"` makes them available per-recording later in the pipeline.

In [8]:
from eegdash import EEGDashDataset
from braindecode.datasets import BaseConcatDataset

_shared = dict(
    task="RestingState",
    description_fields=["subject", "session", "run", "task", "age", "sex"],
    cache_dir="./eegdash_data",
)

# EEGDashDataset requires a single dataset ID per instance (used to build the cache path).
# Load each HBN release separately, then merge by flattening their recording lists.
ds1 = EEGDashDataset(dataset="ds005505", **_shared)  # R1
ds2 = EEGDashDataset(dataset="ds005506", **_shared)  # R2

ds = BaseConcatDataset(ds1.datasets + ds2.datasets)

print(f"R1 recordings : {len(ds1.datasets)}")
print(f"R2 recordings : {len(ds2.datasets)}")
print(f"Total         : {len(ds.datasets)}")

# Inspect the first recording
desc = ds.datasets[0].description
print(f"\nSample — subject: {desc.get('subject')}, age: {desc.get('age')}, task: {desc.get('task')}")

╭─────────────────────────────────────── EEG 2025 Competition Data Notice ────────────────────────────────────────╮
│ This notice is only for users who are participating in the ]8;id=774795;https://eeg2025.github.io/\EEG 2025 Competition]8;;\.                                │
│                                                                                                                 │
│ EEG 2025 Competition Data Notice!                                                                               │
│ You are loading one of the datasets that is used in competition, but via `EEGDashDataset`.                      │
│                                                                                                                 │
│ IMPORTANT:                                                                                                      │
│ If you download data from `EEGDashDataset`, it is NOT identical to the official                                 │
│ competition data, which is accessed via `EEGChallengeDataset`. The competition data has been downsampled and    │
│ filtered.                                                                                                       │
│                                                                                                                 │
│ If you are participating in the competition,                                                                    │
│ you must use the `EEGChallengeDataset` object to ensure consistency.                                            │
│                                                                                                                 │
│ If you are not participating in the competition, you can ignore this message.                                   │
╰──────────────────────────────────────────── Source: EEGDashDataset ─────────────────────────────────────────────╯

╭─────────────────────────────────────── EEG 2025 Competition Data Notice ────────────────────────────────────────╮
│ This notice is only for users who are participating in the ]8;id=120595;https://eeg2025.github.io/\EEG 2025 Competition]8;;\.                                │
│                                                                                                                 │
│ EEG 2025 Competition Data Notice!                                                                               │
│ You are loading one of the datasets that is used in competition, but via `EEGDashDataset`.                      │
│                                                                                                                 │
│ IMPORTANT:                                                                                                      │
│ If you download data from `EEGDashDataset`, it is NOT identical to the official                                 │
│ competition data, which is accessed via `EEGChallengeDataset`. The competition data has been downsampled and    │
│ filtered.                                                                                                       │
│                                                                                                                 │
│ If you are participating in the competition,                                                                    │
│ you must use the `EEGChallengeDataset` object to ensure consistency.                                            │
│                                                                                                                 │
│ If you are not participating in the competition, you can ignore this message.                                   │
╰──────────────────────────────────────────── Source: EEGDashDataset ─────────────────────────────────────────────╯

R1 recordings : 136
R2 recordings : 150
Total         : 286

Sample — subject: NDARCJ475WJP, age: 5.2195, task: RestingState


---
## 2. Preprocessing

Raw EEG contains artifacts and noise that must be removed before feature extraction. We apply three standard steps in sequence using braindecode's `Preprocessor` API, which wraps MNE methods and applies them to every recording in the dataset.

- **Notch filter at 60 Hz** — removes power-line interference (North American standard; use 50 Hz for European data)
- **Average re-reference** — re-references all channels to the average across the scalp, reducing electrode-specific noise
- **Bandpass 1–55 Hz** — removes slow drift (below 1 Hz) and high-frequency noise above the notch, keeping the physiologically relevant EEG bands

Setting `n_jobs=-1` parallelises processing across all available CPU cores.

In [ ]:
from braindecode.preprocessing import Preprocessor, preprocess

preprocess(
    ds,
    [
        Preprocessor("notch_filter", freqs=60),
        Preprocessor("set_eeg_reference", ref_channels="average"),
        Preprocessor("filter", l_freq=1.0, h_freq=55.0),
    ],
    n_jobs=-1,
)

# Confirm preprocessing succeeded
sample_raw = ds.datasets[0].raw
print(f"Channels: {len(sample_raw.ch_names)}")
print(f"Sampling rate: {sample_raw.info['sfreq']} Hz")
print(f"First 5 channels: {sample_raw.ch_names[:5]}")

---
## Optional: Artifact Removal (ICA)

> **This section is optional.** The rest of the tutorial runs without it — skip to Section 3 if you want to stay on the core pipeline.

Resting-state EEG is commonly contaminated by eye blinks and muscle activity. Independent Component Analysis (ICA) decomposes the signal into statistically independent components so you can remove the ones that correspond to non-neural sources.

We wrap MNE's ICA in a plain function and pass it to braindecode's `Preprocessor`, which applies it to every recording automatically. Fitting ICA is CPU-intensive; use `n_jobs=1` here because MNE's ICA already uses multithreading internally.

**Automated labelling (optional):** If `mne-icalabel` is installed (`pip install mne-icalabel`), the commented-out block uses a pre-trained classifier to automatically identify and exclude eye and muscle components — no manual inspection needed.

In [ ]:
from mne.preprocessing import ICA
from braindecode.preprocessing import Preprocessor, preprocess

def apply_ica(raw, n_components=20, random_state=42):
    """Fit ICA and remove artifactual components in-place."""
    ica = ICA(n_components=n_components, random_state=random_state, max_iter="auto")
    ica.fit(raw)

    # --- Automated labelling via mne-icalabel (optional) ---
    # Requires: pip install mne-icalabel
    # from mne_icalabel import label_components
    # labels = label_components(raw, ica, method="iclabel")
    # exclude = [i for i, lbl in enumerate(labels["labels"])
    #            if lbl not in ("brain", "other")]
    # ica.exclude = exclude

    ica.apply(raw)
    return raw

# Apply ICA to every recording (n_jobs=1 recommended — MNE ICA is already multithreaded)
preprocess(ds, [Preprocessor(apply_ica, n_components=20)], n_jobs=1)

print("ICA applied. Checking first recording:")
print(f"  Channels: {len(ds.datasets[0].raw.ch_names)}")  # channel count unchanged

---
## 3. Windowing

Machine learning models expect fixed-size inputs, so we segment each continuous recording into overlapping windows. We use 4-second windows (1024 samples at 256 Hz) with 50% overlap (stride = 512 samples). The 50% overlap is a common trade-off: it roughly doubles the number of training examples while keeping adjacent windows highly correlated.

Setting `drop_last_window=True` discards any final window shorter than 1024 samples, ensuring all windows have identical length.

In [10]:
from braindecode.preprocessing import create_fixed_length_windows

windows_ds = create_fixed_length_windows(
    ds,
    window_size_samples=1024,   # 4 s at 256 Hz
    window_stride_samples=512,  # 50% overlap
    drop_last_window=True,
)

total_windows = sum(len(d) for d in windows_ds.datasets)
print(f"Total windows across all recordings: {total_windows}")
print(f"Windows in first recording: {len(windows_ds.datasets[0])}")

[04/21/26 14:40:32] WARNING  File not found on S3, skipping:                                      ]8;id=357326;file:///Users/mymac/vscode/EEGDash/eegdash/downloader.py\downloader.py]8;;\:]8;id=181824;file:///Users/mymac/vscode/EEGDash/eegdash/downloader.py#146\146]8;;\
                             s3://openneuro.org/ds005505/sub-NDARCJ475WJP/eeg/sub-NDARCJ475WJP_ta                  
                             sk-RestingState_eeg.fdt                                                               

KeyboardInterrupt: 

---
## 4. Feature Extraction

Raw EEG windows are high-dimensional and non-stationary. We summarise each window with two families of interpretable features:

- **Spectral band power** — average power in the delta (1–4.5 Hz), theta (4.5–8 Hz), alpha (8–12 Hz), and beta (12–30 Hz) bands for each channel. These bands are strongly linked to cognitive state and age-related changes.
- **Magnitude-squared coherence (MSC)** — a frequency-resolved measure of synchronisation between pairs of channels. High inter-regional coherence in alpha is a known marker of healthy ageing.

The nested `FeatureExtractor` structure allows each feature group to have its own preprocessor (e.g., FFT for spectral features, cross-spectral density for coherence). `extract_features` returns a `FeaturesConcatDataset` whose `.datasets` list mirrors the structure of `windows_ds`.

In [ ]:
from eegdash.features import (
    FeatureExtractor,
    extract_features,
    spectral_preprocessor,
    spectral_bands_power,
    connectivity_coherency_preprocessor,
    connectivity_magnitude_square_coherence,
)

extractor = FeatureExtractor({
    "spectral": FeatureExtractor(
        {"bands": spectral_bands_power},
        preprocessor=spectral_preprocessor,
    ),
    "connectivity": FeatureExtractor(
        {"msc": connectivity_magnitude_square_coherence},
        preprocessor=connectivity_coherency_preprocessor,
    ),
})

features_ds = extract_features(windows_ds, extractor, batch_size=512, n_jobs=-1)

In [ ]:
# Inspect the feature output
sample_feat_ds = features_ds.datasets[0]
feat_df = sample_feat_ds.features

print(f"Feature matrix shape (windows x features): {feat_df.shape}")
print(f"\nFirst 8 feature column names:")
for col in feat_df.columns[:8]:
    print(f"  {col}")
print("  ...")

---
## 5. Data Preparation — Preventing Data Leakage

This step is critical and easy to get wrong. Because we created many overlapping windows from each recording, a naive random split of windows would place windows from the **same subject** in both train and test sets. The model would then "memorise" individual subjects' EEG signatures rather than learning generalisable patterns — a form of data leakage that inflates test performance.

**The correct approach:** split at the subject level so that all windows from a given subject appear exclusively in either train or test. We also reduce each subject to a single row by averaging features across their windows, which further removes any window-level correlation.

Each `FeaturesDataset` in `features_ds.datasets` corresponds to one recording. We read the subject ID and age from `.description` (recording-level metadata), then average the per-window features from `.features`.

In [ ]:
import pandas as pd
import numpy as np

rows = []
for feat_ds in features_ds.datasets:
    subj = feat_ds.description.get("subject")
    age = feat_ds.description.get("age")
    if age is None:
        continue
    # Average features across all windows for this recording
    mean_feats = feat_ds.features.mean(axis=0)
    row = {"subject": subj, "age": float(age)}
    row.update(mean_feats.to_dict())
    rows.append(row)

subject_df = pd.DataFrame(rows).dropna()
print(f"Subjects with valid age labels: {len(subject_df)}")
print(f"Age range: {subject_df['age'].min():.0f} – {subject_df['age'].max():.0f} years")
subject_df[["subject", "age"]].head()

In [ ]:
from sklearn.model_selection import train_test_split

X = subject_df.drop(columns=["subject", "age"]).values
y = subject_df["age"].values

# Split at the subject level — each subject appears in only one split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training subjects: {X_train.shape[0]}")
print(f"Test subjects:     {X_test.shape[0]}")
print(f"Features per subject: {X_train.shape[1]}")

---
## 6. LightGBM Age Prediction

We use a gradient-boosted decision tree model (LightGBM) to predict age from the EEG features. LightGBM handles high-dimensional tabular data well without requiring feature scaling and natively handles feature importance. We report **Mean Absolute Error (MAE)** — the average error in years — and **R²** — the proportion of age variance explained by the model.

> **Note:** These two datasets together contain a relatively small number of subjects. The metrics below are intended to illustrate the pipeline, not to represent publication-quality predictive performance. Larger cohorts and additional preprocessing (e.g., artefact rejection, ICA) would be needed for a rigorous study.

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, r2_score

model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Test MAE: {mae:.2f} years")
print(f"Test R²:  {r2:.3f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.7, edgecolors="k", linewidths=0.5, s=60)

# Perfect prediction line
lims = [min(y_test.min(), y_pred.min()) - 2, max(y_test.max(), y_pred.max()) + 2]
ax.plot(lims, lims, "--", color="gray", linewidth=1.5, label="Perfect prediction")

ax.set_xlabel("Actual age (years)", fontsize=12)
ax.set_ylabel("Predicted age (years)", fontsize=12)
ax.set_title(f"Age Prediction: MAE = {mae:.2f} yr,  R² = {r2:.3f}", fontsize=12)
ax.legend(fontsize=10)
ax.set_xlim(lims)
ax.set_ylim(lims)
plt.tight_layout()
plt.show()

---
## 7. Extension: SHAP Feature Importance (Optional)

SHAP (SHapley Additive exPlanations) attributes each prediction to individual features in a theoretically principled way. The beeswarm plot below shows which EEG features drive age predictions: each dot is one test subject, coloured by feature value, positioned horizontally by its SHAP contribution.

This step requires the `shap` package (`pip install shap`) and adds significant computation time on large feature sets. It is optional — the rest of the tutorial is fully self-contained without it.

In [ ]:
# Optional — requires: pip install shap
try:
    import shap

    feature_names = subject_df.drop(columns=["subject", "age"]).columns.tolist()

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)

    plt.figure()
    shap.summary_plot(
        shap_values,
        X_test,
        feature_names=feature_names,
        plot_type="dot",
        max_display=20,
        show=True,
    )
except ImportError:
    print("shap is not installed. Run: pip install shap")

---
## Summary

In this tutorial you built a complete EEG-to-prediction pipeline:

1. **Loaded** resting-state EEG recordings from two open-access BIDS datasets via `EEGDashDataset`
2. **Preprocessed** the data with notch filtering, average re-referencing, and bandpass filtering
3. **Windowed** continuous recordings into 4-second epochs with 50% overlap
4. **Extracted** spectral band power and inter-channel coherence features
5. **Aggregated** features at the subject level and split correctly to avoid leakage
6. **Trained** a LightGBM regressor and evaluated it with MAE and R²
7. **Interpreted** predictions with SHAP

**Next steps:** Try Tutorial 2 for classification tasks, or explore `eegdash.features` for additional feature types such as Hjorth parameters and non-linear complexity measures.